# Análisis Exploratorio de Egresos Hospitalarios en Chile (2019–2024)

**Fuente de datos:** Grupos Relacionados por el Diagnóstico (GRD) del sector público, publicados por FONASA.  
**Período:** 2019 – 2024 (~5.8 millones de egresos).  
**Objetivo:** Describir el perfil demográfico, clínico y operativo de las hospitalizaciones públicas en Chile, identificar patrones temporales (incluido el impacto de COVID-19) y evaluar la variabilidad hospitalaria en la práctica clínica.

---
**Notas técnicas del dataset:**
- Los archivos 2019–2021 están en UTF-8; 2022–2023 en UTF-16; 2024 en Latin-1.
- En 2024, la variable `CIP_ENCRIPTADO` se llama `ID_BENEFICIARIO`.
- La variable `EDAD` no existe directamente; se calcula como `FECHA_INGRESO − FECHA_NACIMIENTO`.
- El separador de campos es `|` (pipe).


## Pregunta de Investigación e Hipótesis

**Pregunta de investigación:**  
¿En qué medida el hospital donde se atiende un paciente determina sus días de estada y la intensidad de procedimientos, incluso cuando los pacientes presentan diagnóstico y severidad clínica comparables?

**Hipótesis:**  
Existe heterogeneidad estadísticamente significativa en resultados asistenciales (días de estada y cantidad de procedimientos) entre hospitales del sistema público chileno, incluso manteniendo constante la severidad y el diagnóstico del paciente mediante el código GRD.

### Variables del modelo analítico

| Rol | Variable | Columna en datos |
|-----|----------|------------------|
| **Independiente** | Hospital | `COD_HOSPITAL` |
| **Dependiente** | Días de estada | `dias_estada` (derivada) |
| **Dependiente** | Cantidad de procedimientos | `n_procedimientos` (derivada) |
| **Control** | Diagnóstico principal (CIE-10) | `DIAGNOSTICO1` |
| **Control** | Código GRD | `IR_29301_COD_GRD` |
| **Control** | Peso relativo GRD | `peso_grd` |
| **Control** | Severidad clínica | `severidad` |
| **Control** | Sexo | `SEXO` |
| **Control** | Edad | `edad` (derivada) |

### Estrategia de análisis
1. Cargar y consolidar los CSV 2019–2024 (6 archivos, ~5.8 M registros).
2. Limpiar datos: fechas, edad, días de estada, outliers (p99).
3. Enriquecer con CIE-10 para obtener descripciones legibles.
4. Análisis exploratorio: demografía, diagnósticos, estancia, complejidad, estacionalidad.
5. Análisis de variabilidad hospitalaria: GRD focal, intervalos de confianza, Kruskal-Wallis, post-hoc Dunn, y consistencia multi-GRD.

## 0. Configuración y Librerías

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)

# Estilo global
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (14, 6),
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "font.family": "sans-serif",
})

COLORES = sns.color_palette("colorblind", 10)
print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


In [2]:
# Columnas que necesitamos para el análisis
COLS_NEEDED = [
    "COD_HOSPITAL", "SEXO", "FECHA_NACIMIENTO", "ETNIA",
    "PROVINCIA", "COMUNA", "NACIONALIDAD", "PREVISION",
    "SERVICIO_SALUD", "TIPO_INGRESO", "ESPECIALIDAD_MEDICA",
    "TIPO_ACTIVIDAD", "FECHA_INGRESO", "FECHAALTA", "TIPOALTA",
    "DIAGNOSTICO1", "IR_29301_COD_GRD", "IR_29301_PESO",
    "IR_29301_SEVERIDAD", "IR_29301_MORTALIDAD",
] + [f"PROCEDIMIENTO{i}" for i in range(1, 31)] + [f"DIAGNOSTICO{i}" for i in range(1, 36)]

# Eliminar duplicados manteniendo orden
COLS_NEEDED = list(dict.fromkeys(COLS_NEEDED))


def load_grd_csv(path, year_label):
    """Carga un CSV GRD detectando encoding automáticamente."""
    src = Path(path)
    raw = src.read_bytes()

    for enc in ["utf-8-sig", "utf-8", "utf-16", "utf-16-le", "latin-1", "cp1252"]:
        try:
            text = raw.decode(enc)
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
    else:
        text = raw.decode("utf-8", errors="replace")
        enc = "utf-8(replace)"

    from io import StringIO
    df = pd.read_csv(
        StringIO(text), sep="|", engine="python",
        on_bad_lines="skip", dtype=str, low_memory=False,
    )
    df.columns = df.columns.str.strip()

    # Estandarizar: ID_BENEFICIARIO → CIP_ENCRIPTADO (2024)
    if "ID_BENEFICIARIO" in df.columns and "CIP_ENCRIPTADO" not in df.columns:
        df.rename(columns={"ID_BENEFICIARIO": "CIP_ENCRIPTADO"}, inplace=True)

    # Seleccionar solo columnas disponibles
    cols_available = [c for c in COLS_NEEDED if c in df.columns]
    df = df[cols_available].copy()
    df["anio_archivo"] = year_label

    print(f"  {src.name:45s} enc={enc:12s}  filas={len(df):>10,}  cols={len(cols_available)}")
    return df


FILES = [
    ("DATASET-PROBLEMA8/GRD_PUBLICO_2019.csv", 2019),
    ("DATASET-PROBLEMA8/GRD_PUBLICO_2020.csv", 2020),
    ("DATASET-PROBLEMA8/GRD_PUBLICO_2021.csv", 2021),
    ("DATASET-PROBLEMA8/GRD_PUBLICO_EXTERNO_2022.csv", 2022),
    ("DATASET-PROBLEMA8/GRD_PUBLICO_2023.csv", 2023),
    ("DATASET-PROBLEMA8/GRD_PUBLICO_2024.csv", 2024),
]

print("Cargando archivos GRD...\n")
dfs = []
for path, year in FILES:
    dfs.append(load_grd_csv(path, year))

df = pd.concat(dfs, axis=0, ignore_index=True)
del dfs  # liberar memoria

print(f"\nTotal de registros cargados: {len(df):,}")
print(f"Columnas: {len(df.columns)}")


Cargando archivos GRD...



ValueError: The 'low_memory' option is not supported with the 'python' engine

## 2. Limpieza y Preparación de Datos

Se realizan las siguientes transformaciones:
1. **Parseo de fechas** y cálculo de **edad** y **días de estada**.
2. **Conteo de diagnósticos y procedimientos** por episodio.
3. **Conversión del peso GRD** (coma → punto decimal).
4. **Eliminación de valores atípicos extremos** (percentil 99 de días de estada).
5. **Enriquecimiento con CIE-10** (nombres legibles de diagnósticos).


In [ ]:
n_inicial = len(df)

# --- Fechas ---
for col_fecha in ["FECHA_INGRESO", "FECHAALTA", "FECHA_NACIMIENTO"]:
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")

# --- Año y mes de ingreso ---
df["anio"] = df["FECHA_INGRESO"].dt.year
df["mes"] = df["FECHA_INGRESO"].dt.month

# --- Edad (en años) ---
df["edad"] = (df["FECHA_INGRESO"] - df["FECHA_NACIMIENTO"]).dt.days / 365.25
df.loc[df["edad"] < 0, "edad"] = np.nan
df.loc[df["edad"] > 120, "edad"] = np.nan

# --- Días de estada ---
df["dias_estada"] = (df["FECHAALTA"] - df["FECHA_INGRESO"]).dt.days
df.loc[df["dias_estada"] < 0, "dias_estada"] = np.nan

# --- Conteo de procedimientos y diagnósticos ---
proc_cols = [c for c in df.columns if c.startswith("PROCEDIMIENTO")]
diag_cols = [c for c in df.columns if c.startswith("DIAGNOSTICO")]
df["n_procedimientos"] = df[proc_cols].notna().sum(axis=1)
df["n_diagnosticos"] = df[diag_cols].notna().sum(axis=1)

# --- Peso GRD (coma decimal → float) ---
df["peso_grd"] = (
    df["IR_29301_PESO"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# --- Severidad y mortalidad como numérico ---
df["severidad"] = pd.to_numeric(df["IR_29301_SEVERIDAD"], errors="coerce")
df["mortalidad"] = pd.to_numeric(df["IR_29301_MORTALIDAD"], errors="coerce")

# --- Eliminar outliers extremos de días de estada (p99) ---
p99 = df["dias_estada"].quantile(0.99)
df_clean = df[df["dias_estada"].notna() & (df["dias_estada"] <= p99)].copy()

# --- Liberar columnas intermedias de memoria ---
cols_drop = proc_cols[1:] + diag_cols[1:]  # Mantener DIAGNOSTICO1 y PROCEDIMIENTO1
df_clean.drop(columns=[c for c in cols_drop if c in df_clean.columns], inplace=True, errors="ignore")

n_final = len(df_clean)
print(f"Registros iniciales:   {n_inicial:>10,}")
print(f"Registros tras limpieza: {n_final:>10,}")
print(f"Eliminados:            {n_inicial - n_final:>10,} ({(n_inicial - n_final)/n_inicial*100:.1f}%)")
print(f"Corte outlier p99 (días estada): {p99:.0f} días")
print(f"\nRango de edad: {df_clean['edad'].min():.0f} – {df_clean['edad'].max():.0f} años")
print(f"Rango de días de estada: {df_clean['dias_estada'].min():.0f} – {df_clean['dias_estada'].max():.0f}")


In [ ]:
# Cargar CIE-10 para enriquecer los diagnósticos
cie10 = pd.read_excel("DATASET-PROBLEMA8/CIE-10.xlsx", dtype=str)
cie10_map = cie10.set_index("Código")["Descripción"].to_dict()
# También crear mapa de capítulos
cie10_capitulo = cie10.set_index("Código")["Capítulo"].to_dict()

# Agregar descripción del diagnóstico principal
df_clean["diag_principal"] = df_clean["DIAGNOSTICO1"].astype(str).str.strip()
df_clean["diag_descripcion"] = df_clean["diag_principal"].map(cie10_map)
df_clean["diag_capitulo"] = df_clean["diag_principal"].map(cie10_capitulo)

cobertura = df_clean["diag_descripcion"].notna().mean() * 100
print(f"Cobertura CIE-10 para diagnóstico principal: {cobertura:.1f}%")
display(df_clean[["diag_principal", "diag_descripcion", "diag_capitulo"]].dropna().head(5))


In [ ]:
# Resumen del dataset limpio
print("=== RESUMEN DEL DATASET LIMPIO ===\n")
print(f"Filas: {len(df_clean):,}")
print(f"Período: {df_clean['anio'].min():.0f} – {df_clean['anio'].max():.0f}")
print(f"Hospitales únicos: {df_clean['COD_HOSPITAL'].nunique():,}")
print(f"Servicios de salud: {df_clean['SERVICIO_SALUD'].nunique()}")
print()
print("Variables numéricas clave:")
display(df_clean[["edad", "dias_estada", "n_procedimientos", "n_diagnosticos", "peso_grd", "severidad"]].describe().round(2))


## Glosario de Términos

| Término | Definición |
|---------|------------|
| **GRD** | Grupos Relacionados por el Diagnóstico. Sistema de clasificación que agrupa egresos hospitalarios en categorías clínicamente coherentes y de consumo de recursos similar. |
| **IR-GRD** | Versión chilena del sistema GRD utilizada por FONASA, adaptada a la realidad epidemiológica y de costos del sistema público nacional. |
| **CIE-10** | Clasificación Internacional de Enfermedades, 10ª revisión. Nomenclatura estándar para diagnósticos médicos. |
| **CIE-9** | Clasificación Internacional de Enfermedades, 9ª revisión. Utilizada para codificar procedimientos médicos e intervenciones quirúrgicas. |
| **Peso Relativo GRD** | Valor numérico que representa la complejidad relativa y el consumo esperado de recursos de un grupo diagnóstico específico. Un peso mayor indica mayor complejidad y costo. |
| **Severidad** | Nivel de gravedad clínica del caso (1–4), precalculado en el sistema IR-GRD. Un nivel más alto indica mayor complejidad clínica. |
| **Mortalidad** | Riesgo de mortalidad asociado al caso (1–4), precalculado en el sistema IR-GRD. |
| **Días de Estada** | Duración total de la hospitalización, calculada como la diferencia entre la fecha de alta (`FECHAALTA`) y la fecha de ingreso (`FECHA_INGRESO`). |
| **FONASA** | Fondo Nacional de Salud. Organismo público de financiamiento de salud en Chile. Tramos A y B corresponden a la población de menores ingresos. |
| **SERVICIO_SALUD** | Red territorial de prestadores de salud pública en Chile. El país se divide en 29 Servicios de Salud regionales. |
| **Egreso hospitalario** | Todo evento de salida de un paciente de una cama hospitalaria, ya sea por alta médica, traslado, fallecimiento u otra causa. |
| **CIP Encriptado / ID_BENEFICIARIO** | Identificador encriptado del paciente (anonimizado). En archivos 2019–2023 se llama `CIP_ENCRIPTADO`; en 2024 se llama `ID_BENEFICIARIO`. |
| **Variabilidad hospitalaria** | Diferencias en resultados asistenciales (días de estada, procedimientos) entre hospitales para pacientes con características clínicas comparables, no explicadas por la condición del paciente.

## 3. Panorama General de los Egresos Hospitalarios

### 3.1 Volumen de egresos hospitalarios por año

In [ ]:
vol = df_clean.groupby("anio").size().reset_index(name="egresos")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(vol["anio"], vol["egresos"], color=COLORES[0], width=0.6, edgecolor="white", linewidth=1.2)
ax.plot(vol["anio"], vol["egresos"], "o-", color=COLORES[1], linewidth=2.5, markersize=8, zorder=5)

# Etiquetas sobre las barras
for bar, val in zip(bars, vol["egresos"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15000,
            f"{val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Anotación COVID
ax.annotate("Impacto COVID-19", xy=(2020, vol[vol.anio==2020]["egresos"].values[0]),
            xytext=(2020.6, vol["egresos"].max() * 0.65),
            arrowprops=dict(arrowstyle="->", color="red", lw=2),
            fontsize=11, color="red", fontweight="bold")

ax.set_xlabel("Año")
ax.set_ylabel("Cantidad de egresos")
ax.set_title("Egresos hospitalarios por año en el sistema público de Chile (2019–2024)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xticks(vol["anio"])
sns.despine()
plt.tight_layout()
plt.show()

# Calcular caída COVID
pre_covid = vol[vol.anio == 2019]["egresos"].values[0]
covid_year = vol[vol.anio == 2020]["egresos"].values[0]
pct_drop = (covid_year - pre_covid) / pre_covid * 100
print(f"Caída 2020 vs 2019: {pct_drop:+.1f}%")


**Conclusión 3.1 — Impacto de COVID-19 y recuperación:**  
El año 2020 registró una caída significativa en los egresos hospitalarios respecto a 2019, directamente atribuible a la pandemia de COVID-19: se suspendieron cirugías programadas, se limitaron las consultas electivas y se reconvirtieron camas para pacientes críticos. A partir de 2021 se observa una recuperación gradual, con 2023 y 2024 superando los niveles pre-pandemia, lo que sugiere tanto una demanda acumulada (cirugías postergadas) como un crecimiento vegetativo de la actividad hospitalaria.


### 3.2 Distribución demográfica: Edad y Sexo

In [ ]:
df_demo = df_clean.dropna(subset=["edad", "SEXO"]).copy()
df_demo = df_demo[df_demo["SEXO"].isin(["HOMBRE", "MUJER"])]
df_demo["grupo_edad"] = pd.cut(df_demo["edad"],
    bins=list(range(0, 101, 5)) + [120],
    labels=[f"{i}-{i+4}" for i in range(0, 100, 5)] + ["100+"],
    right=False
)

# Pirámide poblacional
pyr = df_demo.groupby(["grupo_edad", "SEXO"], observed=True).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = range(len(pyr))
labels = pyr.index.astype(str)

ax.barh(y_pos, -pyr["HOMBRE"], color=COLORES[0], label="Hombre", height=0.8, edgecolor="white")
ax.barh(y_pos, pyr["MUJER"], color=COLORES[3], label="Mujer", height=0.8, edgecolor="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Cantidad de egresos")
ax.set_ylabel("Grupo de edad")
ax.set_title("Pirámide de egresos hospitalarios por edad y sexo (2019–2024)")
ax.legend(loc="lower right", fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):,.0f}"))
ax.axvline(0, color="black", linewidth=0.5)
sns.despine()
plt.tight_layout()
plt.show()

print(f"Distribución por sexo: {df_demo['SEXO'].value_counts().to_dict()}")
print(f"Edad promedio — Mujeres: {df_demo[df_demo.SEXO=='MUJER']['edad'].mean():.1f} años, "
      f"Hombres: {df_demo[df_demo.SEXO=='HOMBRE']['edad'].mean():.1f} años")


**Conclusión 3.2 — Perfil demográfico:**  
La pirámide de egresos muestra un claro predominio femenino en las edades de 20 a 39 años, explicado por las hospitalizaciones obstétricas (embarazo, parto y puerperio). En adultos mayores (65+), la distribución se equilibra y refleja la carga de enfermedad crónica. El grupo pediátrico (0-4 años) también presenta un volumen relevante, asociado a patologías neonatales y respiratorias. Esta distribución confirma que los hospitales públicos chilenos atienden principalmente población en edad reproductiva y adultos mayores con multimorbilidad.


### 3.3 Distribución por previsión de salud

In [ ]:
prev = df_clean["PREVISION"].value_counts().head(10).reset_index()
prev.columns = ["prevision", "n"]
prev["pct"] = (prev["n"] / prev["n"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(prev["prevision"][::-1], prev["n"][::-1], color=COLORES[2], edgecolor="white")
for bar, pct in zip(bars, prev["pct"][::-1]):
    ax.text(bar.get_width() + prev["n"].max()*0.01, bar.get_y() + bar.get_height()/2,
            f"{pct:.1f}%", va="center", fontsize=10)

ax.set_xlabel("Cantidad de egresos")
ax.set_title("Distribución de egresos por previsión de salud")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 3.3 — Perfil socioeconómico de los usuarios:**  
Los tramos FONASA A y B (personas de menores ingresos y carentes de recursos) concentran la gran mayoría de los egresos, lo que confirma que el sistema público hospitalario chileno atiende predominantemente a la población más vulnerable socioeconómicamente. Los usuarios de ISAPRE y particulares representan una fracción marginal, ya que estos segmentos se atienden principalmente en el sector privado. Esta concentración tiene implicancias para la asignación de recursos: las políticas de financiamiento hospitalario deben considerar que la población atendida tiene alta vulnerabilidad social.


### 3.4 Nacionalidad de los pacientes

In [ ]:
nac = df_clean["NACIONALIDAD"].value_counts().head(12).reset_index()
nac.columns = ["nacionalidad", "n"]
nac["pct"] = (nac["n"] / len(df_clean) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 2]})

# Pie: Chile vs extranjero
chile_n = df_clean[df_clean["NACIONALIDAD"].str.upper().str.strip() == "CHILE"].shape[0]
extran_n = len(df_clean) - chile_n
axes[0].pie([chile_n, extran_n], labels=["Chile", "Extranjero"],
            autopct="%1.1f%%", colors=[COLORES[0], COLORES[3]], startangle=90,
            textprops={"fontsize": 12})
axes[0].set_title("Proporción Chile vs Extranjero")

# Bar: top nacionalidades extranjeras
nac_extran = nac[nac["nacionalidad"].str.upper().str.strip() != "CHILE"].head(10)
axes[1].barh(nac_extran["nacionalidad"][::-1], nac_extran["n"][::-1], color=COLORES[4], edgecolor="white")
for i, (_, row) in enumerate(nac_extran[::-1].iterrows()):
    axes[1].text(row["n"] + nac_extran["n"].max()*0.02, i, f"{row['pct']:.2f}%", va="center", fontsize=9)
axes[1].set_xlabel("Cantidad de egresos")
axes[1].set_title("Top nacionalidades extranjeras")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine(ax=axes[1])

plt.suptitle("Nacionalidad de los pacientes hospitalizados (2019–2024)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


**Conclusión 3.4 — Migración y demanda hospitalaria:**  
La inmensa mayoría de los egresos corresponde a pacientes de nacionalidad chilena. Sin embargo, se observa una presencia creciente de pacientes extranjeros, principalmente de países como Haití, Perú, Venezuela, Bolivia y Colombia, reflejando los flujos migratorios recientes hacia Chile. Aunque su proporción es pequeña respecto al total, este dato es relevante para la planificación de servicios interculturales y la adecuación de protocolos de atención.


### 3.5 Distribución por etnia

In [ ]:
etnia = df_clean.dropna(subset=["ETNIA"]).copy()
etnia_counts = etnia["ETNIA"].value_counts().reset_index()
etnia_counts.columns = ["etnia", "n"]
etnia_counts["pct"] = (etnia_counts["n"] / len(df_clean) * 100).round(2)

# Separar "sin etnia / no indígena" del resto para mejor visualización
mask_indig = ~etnia_counts["etnia"].str.upper().str.contains("SIN|NO INDÍGENA|NO INDIGENA|NINGUNO", na=False)
etnia_indig = etnia_counts[mask_indig].head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pie: indígena vs no indígena
n_indig = etnia_indig["n"].sum()
n_no_indig = len(df_clean) - n_indig
axes[0].pie(
    [n_indig, n_no_indig],
    labels=["Pueblo originario", "Sin etnia / No declarado"],
    autopct="%1.1f%%",
    colors=[COLORES[3], COLORES[0]],
    startangle=90,
    textprops={"fontsize": 11}
)
axes[0].set_title("Proporción indígena vs no indígena")

# Barra: top pueblos originarios
if len(etnia_indig) > 0:
    colors_et = sns.color_palette("Set2", len(etnia_indig))
    axes[1].barh(
        etnia_indig["etnia"][::-1],
        etnia_indig["n"][::-1],
        color=colors_et[::-1], edgecolor="white"
    )
    for i, (_, row) in enumerate(etnia_indig[::-1].iterrows()):
        axes[1].text(
            row["n"] + etnia_indig["n"].max() * 0.02, i,
            f"{row['pct']:.2f}%", va="center", fontsize=9
        )
    axes[1].set_xlabel("Cantidad de egresos")
    axes[1].set_title("Egresos por pueblo originario")
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    sns.despine(ax=axes[1])
else:
    axes[1].text(0.5, 0.5, "Sin datos de pueblos originarios", ha="center", va="center")

plt.suptitle("Distribución étnica de pacientes hospitalizados (2019–2024)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Registros con etnia declarada: {len(etnia):,} ({len(etnia)/len(df_clean)*100:.1f}% del total)")
print(f"\nTop etnias registradas:")
print(etnia_counts.head(12).to_string(index=False))

**Conclusión 3.5 — Diversidad étnica de los pacientes:**  
La distribución étnica muestra que la gran mayoría de los pacientes hospitalizados pertenece al grupo no indígena o no declaró etnia. Sin embargo, la presencia de pueblos originarios — principalmente Mapuche, Aymara, Atacameño y otros — en la red hospitalaria pública es relevante desde la perspectiva de la pertinencia cultural de la atención. Esta información es esencial para el diseño de programas de salud intercultural, la evaluación de brechas de acceso diferenciadas y el cumplimiento de la Ley 19.253 de pueblos indígenas en el ámbito sanitario.

## 4. Perfil Clínico de las Hospitalizaciones

### 4.1 Diagnósticos principales más frecuentes

In [ ]:
top_diag = (
    df_clean.groupby(["diag_principal", "diag_descripcion"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
    .head(20)
)
top_diag["label"] = top_diag["diag_principal"] + " – " + top_diag["diag_descripcion"].fillna("Sin descripción")
top_diag["label"] = top_diag["label"].str[:65]  # truncar etiquetas largas

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(top_diag["label"][::-1], top_diag["n"][::-1], color=COLORES[0], edgecolor="white")
for i, val in enumerate(top_diag["n"][::-1]):
    ax.text(val + top_diag["n"].max()*0.01, i, f"{val:,.0f}", va="center", fontsize=9)

ax.set_xlabel("Cantidad de egresos")
ax.set_title("Top 20 diagnósticos principales (CIE-10) en egresos hospitalarios")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 4.1 — Diagnósticos predominantes:**  
Los diagnósticos más frecuentes revelan el perfil epidemiológico de la hospitalización pública chilena. Las causas obstétricas (parto normal, cesárea), la patología quirúrgica abdominal (colelitiasis/colecistitis) y las cataratas dominan el ranking. La presencia de neumonía (J18.9) y de infecciones urinarias (N39.0) refleja la carga de patología infecciosa, especialmente en adultos mayores. Los códigos Z (quimioterapia, diálisis) indican un volumen significativo de hospitalizaciones recurrentes por enfermedades crónicas, que consumen recursos de forma repetida.


### 4.2 Especialidades médicas más frecuentes

In [ ]:
esp = df_clean["ESPECIALIDAD_MEDICA"].value_counts().head(15).reset_index()
esp.columns = ["especialidad", "n"]
esp["pct"] = (esp["n"] / len(df_clean) * 100).round(1)

fig, ax = plt.subplots(figsize=(14, 6))
colors_esp = sns.color_palette("viridis", len(esp))
ax.barh(esp["especialidad"][::-1], esp["n"][::-1], color=colors_esp[::-1], edgecolor="white")
for i, (_, row) in enumerate(esp[::-1].iterrows()):
    ax.text(row["n"] + esp["n"].max()*0.01, i, f"{row['pct']:.1f}%", va="center", fontsize=9)

ax.set_xlabel("Cantidad de egresos")
ax.set_title("Top 15 especialidades médicas por volumen de egresos")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 4.2 — Demanda por especialidad:**  
Obstetricia y Ginecología, Cirugía General y Medicina Interna constituyen las tres especialidades con mayor volumen de egresos, concentrando aproximadamente la mitad del total. Este patrón refleja la estructura demográfica (alta natalidad relativa y envejecimiento progresivo) y la carga quirúrgica del sistema. Traumatología, Oftalmología y Pediatría completan el grupo de alta demanda. La distribución sugiere que las políticas de formación de especialistas y la planificación de pabellones deben priorizar estas áreas.


### 4.3 Tipo de ingreso por año

In [ ]:
tipo_anio = (
    df_clean.groupby(["anio", "TIPO_INGRESO"])
    .size()
    .reset_index(name="n")
)
tipo_anio = tipo_anio[tipo_anio["TIPO_INGRESO"].isin(["URGENCIA", "PROGRAMADA", "OBSTETRICA"])]

pivot = tipo_anio.pivot(index="anio", columns="TIPO_INGRESO", values="n").fillna(0)

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind="bar", stacked=True, ax=ax,
           color=[COLORES[1], COLORES[3], COLORES[0]], edgecolor="white", width=0.7)
ax.set_xlabel("Año")
ax.set_ylabel("Cantidad de egresos")
ax.set_title("Tipo de ingreso hospitalario por año")
ax.legend(title="Tipo de ingreso", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
sns.despine()
plt.tight_layout()
plt.show()

# Porcentajes
pct_tipo = df_clean["TIPO_INGRESO"].value_counts(normalize=True).mul(100).round(1)
print("Distribución global tipo de ingreso:")
print(pct_tipo.to_string())


**Conclusión 4.3 — Urgencias vs. cirugía programada:**  
Las hospitalizaciones de urgencia representan consistentemente la mayor proporción de ingresos en todos los años, lo que refleja una alta demanda no planificada sobre el sistema. En 2020, la caída afectó especialmente a las cirugías programadas y obstétricas (postergación de intervenciones electivas), mientras que las urgencias mantuvieron un volumen relativamente mayor. La recuperación post-pandemia muestra un aumento en las cirugías programadas, sugiriendo la resolución de listas de espera acumuladas.


### 4.4 Tipo de alta

In [ ]:
alta = df_clean["TIPOALTA"].value_counts().head(10).reset_index()
alta.columns = ["tipo_alta", "n"]
alta["pct"] = (alta["n"] / len(df_clean) * 100).round(2)

fig, ax = plt.subplots(figsize=(14, 5))
colors_alta = [COLORES[0]] * len(alta)
colors_alta[1] = COLORES[1]  # Resaltar fallecido

ax.barh(alta["tipo_alta"][::-1], alta["n"][::-1],
        color=colors_alta[::-1], edgecolor="white")
for i, (_, row) in enumerate(alta[::-1].iterrows()):
    ax.text(row["n"] + alta["n"].max()*0.01, i, f"{row['pct']:.1f}%", va="center", fontsize=9)

ax.set_xlabel("Cantidad de egresos")
ax.set_title("Distribución por tipo de alta hospitalaria")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()

mort = df_clean[df_clean["TIPOALTA"].str.contains("FALLECIDO", na=False)].shape[0]
print(f"Tasa de mortalidad intrahospitalaria: {mort/len(df_clean)*100:.2f}%")


**Conclusión 4.4 — Desenlace de las hospitalizaciones:**  
La gran mayoría de los egresos resultan en alta a domicilio, lo cual es esperable en un sistema donde las estadías son cortas. La tasa de mortalidad intrahospitalaria se ubica en torno al 2-3%, un indicador clave de calidad asistencial. Las derivaciones a otros centros y las fugas de pacientes, aunque minoritarias, representan señales de fragmentación del sistema o de insatisfacción con la atención. La hospitalización domiciliaria, aunque incipiente, muestra una tendencia a la descongestión de camas.


## 5. Análisis de la Estancia Hospitalaria

### 5.1 Distribución global de días de estada

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma
axes[0].hist(df_clean["dias_estada"], bins=80, color=COLORES[0], edgecolor="white", alpha=0.85)
axes[0].axvline(df_clean["dias_estada"].median(), color="red", linestyle="--", linewidth=2, label=f"Mediana: {df_clean['dias_estada'].median():.0f}")
axes[0].axvline(df_clean["dias_estada"].mean(), color="orange", linestyle="--", linewidth=2, label=f"Media: {df_clean['dias_estada'].mean():.1f}")
axes[0].set_xlabel("Días de estada")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de días de estada")
axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Boxplot
bp = axes[1].boxplot(df_clean["dias_estada"].dropna(), vert=False, widths=0.6,
                     patch_artist=True, boxprops=dict(facecolor=COLORES[0], alpha=0.7),
                     medianprops=dict(color="red", linewidth=2))
axes[1].set_xlabel("Días de estada")
axes[1].set_title("Boxplot de días de estada")

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print(f"Estadísticas de días de estada:")
print(f"  Media:   {df_clean['dias_estada'].mean():.2f}")
print(f"  Mediana: {df_clean['dias_estada'].median():.0f}")
print(f"  Moda:    {df_clean['dias_estada'].mode().values[0]:.0f}")
print(f"  Desv. estándar: {df_clean['dias_estada'].std():.2f}")
print(f"  P25:     {df_clean['dias_estada'].quantile(0.25):.0f}")
print(f"  P75:     {df_clean['dias_estada'].quantile(0.75):.0f}")


**Conclusión 5.1 — Perfil de la estancia hospitalaria:**  
La distribución de días de estada es marcadamente asimétrica a la derecha (right-skewed): la mayoría de las hospitalizaciones son cortas (mediana de ~2 días), pero existe una cola larga de pacientes con estancias prolongadas que elevan considerablemente la media. Esta asimetría es típica de datos hospitalarios y tiene implicancias directas: la mediana es más representativa que la media para describir la estancia "típica", y los pacientes de larga estancia (aunque pocos) consumen una proporción desproporcionada de camas-día.


### 5.2 Días de estada por nivel de severidad

In [ ]:
df_sev = df_clean.dropna(subset=["severidad", "dias_estada"]).copy()
df_sev = df_sev[df_sev["severidad"].isin([1, 2, 3])]  # Excluir 0 (sin clasificar)
df_sev["severidad_label"] = df_sev["severidad"].map({1: "1 – Menor", 2: "2 – Moderada", 3: "3 – Mayor"})

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_sev, x="severidad_label", y="dias_estada",
               palette=[COLORES[2], COLORES[4], COLORES[1]], inner="box", cut=0, ax=ax)
ax.set_xlabel("Nivel de severidad (IR-GRD)")
ax.set_ylabel("Días de estada")
ax.set_title("Distribución de días de estada según severidad del episodio")
sns.despine()
plt.tight_layout()
plt.show()

print("Estadísticas por severidad:")
display(df_sev.groupby("severidad_label")["dias_estada"].describe().round(1))


**Conclusión 5.2 — Severidad como predictor de estancia:**  
Existe una relación dosis-respuesta clara entre el nivel de severidad y los días de estada: a mayor severidad, mayor estancia media y mayor dispersión. Los pacientes con severidad 3 (mayor) presentan estancias significativamente más largas y más variables, reflejando la complejidad clínica de estos casos. Esta relación valida el uso del indicador de severidad GRD como variable de ajuste al comparar hospitales, ya que parte importante de la variabilidad en estancia se explica por la complejidad del paciente.


### 5.3 Días de estada por tipo de ingreso

In [ ]:
df_tipo = df_clean[df_clean["TIPO_INGRESO"].isin(["URGENCIA", "PROGRAMADA", "OBSTETRICA"])].copy()

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df_tipo, x="TIPO_INGRESO", y="dias_estada",
            palette=[COLORES[1], COLORES[0], COLORES[3]],
            showfliers=False, ax=ax, width=0.5)
ax.set_xlabel("Tipo de ingreso")
ax.set_ylabel("Días de estada")
ax.set_title("Días de estada según tipo de ingreso")
sns.despine()
plt.tight_layout()
plt.show()

print("Mediana de días de estada por tipo:")
print(df_tipo.groupby("TIPO_INGRESO")["dias_estada"].median().sort_values(ascending=False).to_string())


**Conclusión 5.3 — Tipo de ingreso y estancia:**  
Las hospitalizaciones de urgencia presentan la mayor variabilidad y estancia mediana más alta, lo cual es coherente con la naturaleza impredecible y potencialmente grave de estas admisiones. Las hospitalizaciones programadas (cirugías electivas) tienden a estancias más controladas y predecibles. Las obstétricas son las más cortas, reflejando protocolos estandarizados de atención del parto. Esta distinción es relevante para la gestión de camas: los flujos de urgencia requieren mayor holgura operativa que los programados.


### 5.4 Evolución temporal de la estancia media

In [ ]:
evol = df_clean.groupby("anio")["dias_estada"].agg(["mean", "median", "std"]).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(evol["anio"], evol["mean"], "o-", color=COLORES[1], linewidth=2.5, markersize=8, label="Media")
ax.plot(evol["anio"], evol["median"], "s--", color=COLORES[0], linewidth=2.5, markersize=8, label="Mediana")
ax.fill_between(evol["anio"], evol["mean"] - evol["std"], evol["mean"] + evol["std"],
                alpha=0.15, color=COLORES[1], label="± 1 Desv. Estándar")

for _, row in evol.iterrows():
    ax.text(row["anio"], row["mean"] + 0.3, f"{row['mean']:.2f}", ha="center", fontsize=9, color=COLORES[1])

ax.set_xlabel("Año")
ax.set_ylabel("Días de estada")
ax.set_title("Evolución de la estancia hospitalaria media y mediana (2019–2024)")
ax.legend(fontsize=10)
ax.set_xticks(evol["anio"])
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 5.4 — Tendencia temporal de la estancia:**  
La estancia media muestra variaciones interanuales. Durante 2020, la composición de casos cambió (menos cirugías electivas cortas, más pacientes graves/COVID), lo que puede haber elevado la estancia media. La mediana se mantiene más estable, confirmando que la media es sensible a los valores extremos. La tendencia post-pandemia sugiere una estabilización progresiva de la estancia, aunque cualquier interpretación debe considerar los cambios simultáneos en el case-mix hospitalario.


## 6. Complejidad Clínica y Procedimientos

### 6.1 Procedimientos por egreso

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma de procedimientos
axes[0].hist(df_clean["n_procedimientos"], bins=range(0, 32),
             color=COLORES[4], edgecolor="white", alpha=0.85)
axes[0].axvline(df_clean["n_procedimientos"].mean(), color="red", linestyle="--",
                linewidth=2, label=f"Media: {df_clean['n_procedimientos'].mean():.1f}")
axes[0].set_xlabel("Número de procedimientos")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de procedimientos por egreso")
axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Histograma de diagnósticos
axes[1].hist(df_clean["n_diagnosticos"], bins=range(0, 37),
             color=COLORES[2], edgecolor="white", alpha=0.85)
axes[1].axvline(df_clean["n_diagnosticos"].mean(), color="red", linestyle="--",
                linewidth=2, label=f"Media: {df_clean['n_diagnosticos'].mean():.1f}")
axes[1].set_xlabel("Número de diagnósticos")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de diagnósticos por egreso")
axes[1].legend(fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print(f"Procedimientos — Media: {df_clean['n_procedimientos'].mean():.1f}, Mediana: {df_clean['n_procedimientos'].median():.0f}")
print(f"Diagnósticos   — Media: {df_clean['n_diagnosticos'].mean():.1f}, Mediana: {df_clean['n_diagnosticos'].median():.0f}")


**Conclusión 6.1 — Intensidad de la atención:**  
El promedio de ~6-7 procedimientos por egreso indica un nivel considerable de actividad clínica por hospitalización. La variación es amplia: hay episodios sin procedimientos (observación, diagnóstico) y otros con más de 20 (cirugías complejas, pacientes de UCI). De forma similar, el número de diagnósticos registrados varía entre 1 y 35, donde los episodios con muchos diagnósticos corresponden a pacientes con alta comorbilidad. Esta variabilidad en la complejidad es fundamental para entender las diferencias en costos y resultados entre hospitales.


### 6.2 Relación entre severidad y riesgo de mortalidad

In [ ]:
df_sm = df_clean.dropna(subset=["severidad", "mortalidad"])
df_sm = df_sm[df_sm["severidad"].isin([0, 1, 2, 3]) & df_sm["mortalidad"].isin([0, 1, 2, 3])]

cross = pd.crosstab(df_sm["severidad"], df_sm["mortalidad"], margins=False)
cross_pct = cross.div(cross.sum().sum()) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(cross, annot=True, fmt=",", cmap="YlOrRd", ax=axes[0],
            linewidths=0.5, cbar_kws={"label": "Cantidad"})
axes[0].set_xlabel("Riesgo de mortalidad")
axes[0].set_ylabel("Severidad")
axes[0].set_title("Conteo: Severidad vs Riesgo de Mortalidad")

sns.heatmap(cross_pct, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[1],
            linewidths=0.5, cbar_kws={"label": "% del total"})
axes[1].set_xlabel("Riesgo de mortalidad")
axes[1].set_ylabel("Severidad")
axes[1].set_title("Porcentaje: Severidad vs Riesgo de Mortalidad")

plt.suptitle("Clasificación de egresos según severidad y riesgo de mortalidad (IR-GRD)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


**Conclusión 6.2 — Perfil de riesgo de los pacientes:**  
La mayoría de los egresos se concentra en la intersección de baja severidad y bajo riesgo de mortalidad (severidad 1, mortalidad 1), lo cual es esperable en un sistema donde predominan las cirugías electivas y los partos. Sin embargo, los pacientes con alta severidad y alto riesgo de mortalidad, aunque minoritarios en volumen, representan los casos de mayor complejidad clínica y consumo de recursos. La categoría "0" corresponde a episodios sin clasificación GRD válida. La correlación positiva entre severidad y mortalidad confirma la consistencia del sistema de clasificación IR-GRD utilizado en Chile.


### 6.3 Egresos por capítulo CIE-10 (grandes grupos de enfermedad)

In [ ]:
df_cap = df_clean.dropna(subset=["diag_capitulo"]).copy()
# Limpiar nombre del capítulo
df_cap["cap_short"] = df_cap["diag_capitulo"].str.replace(r"^Cap\.\d+\s+", "", regex=True).str[:70]

cap_count = df_cap["cap_short"].value_counts().head(15).reset_index()
cap_count.columns = ["capitulo", "n"]
cap_count["pct"] = (cap_count["n"] / len(df_cap) * 100).round(1)

fig, ax = plt.subplots(figsize=(14, 8))
colors_cap = sns.color_palette("tab20", len(cap_count))
ax.barh(cap_count["capitulo"][::-1], cap_count["n"][::-1], color=colors_cap[::-1], edgecolor="white")
for i, (_, row) in enumerate(cap_count[::-1].iterrows()):
    ax.text(row["n"] + cap_count["n"].max()*0.01, i, f"{row['pct']:.1f}%", va="center", fontsize=9)

ax.set_xlabel("Cantidad de egresos")
ax.set_title("Egresos por capítulo CIE-10 (Top 15)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 6.3 — Carga de enfermedad por grandes grupos:**  
Los capítulos CIE-10 con mayor representación confirman el perfil epidemiológico esperado: embarazo/parto/puerperio, enfermedades del aparato digestivo (colecistitis, apendicitis), neoplasias (oncología) y enfermedades del aparato circulatorio (cardiopatías). Los factores que influyen en el estado de salud (capítulo Z, que incluye quimioterapia y diálisis) representan un volumen significativo de hospitalizaciones recurrentes. Este panorama es coherente con la transición epidemiológica de Chile, donde coexisten enfermedades crónicas no transmisibles con una carga obstétrica relevante.


## 7. Patrones Temporales y Estacionalidad

### 7.1 Estacionalidad mensual de los egresos

In [ ]:
monthly = df_clean.groupby(["anio", "mes"]).size().reset_index(name="n")

fig, ax = plt.subplots(figsize=(14, 6))
for year in sorted(monthly["anio"].unique()):
    data = monthly[monthly["anio"] == year]
    style = "-" if year != 2020 else "--"
    width = 2.5 if year != 2020 else 3
    color = COLORES[1] if year == 2020 else None
    ax.plot(data["mes"], data["n"], f"o{style}", linewidth=width,
            markersize=5, label=str(int(year)), color=color)

ax.set_xlabel("Mes")
ax.set_ylabel("Cantidad de egresos")
ax.set_title("Estacionalidad mensual de egresos hospitalarios por año")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                     "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"])
ax.legend(title="Año", fontsize=9, ncol=3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 7.1 — Patrones estacionales:**  
Se observa un patrón estacional consistente entre años: caída en febrero (vacaciones de verano, menor actividad quirúrgica programada), aumento en los meses de invierno (junio-agosto, asociado a enfermedades respiratorias), y un descenso en diciembre (fiestas, cierre de pabellones). El año 2020 presenta un patrón completamente distorsionado: caída abrupta en marzo-abril (cuarentenas), recuperación parcial en el segundo semestre. A partir de 2021, los patrones estacionales se normalizan progresivamente, aunque con un volumen base mayor que el pre-pandémico.


## 8. Variabilidad Hospitalaria

Esta sección analiza si existen diferencias significativas en los días de estada entre hospitales para pacientes con el mismo diagnóstico y complejidad comparable. Esta es la pregunta central de investigación: **¿el hospital donde se atiende un paciente influye en su resultado, independientemente de su condición clínica?**


### 8.1 Selección del GRD más frecuente y análisis por hospital

In [ ]:
# Seleccionar el GRD con más casos para un análisis robusto
grd_counts = df_clean["IR_29301_COD_GRD"].value_counts()
target_grd = grd_counts.index[0]
print(f"GRD más frecuente: {target_grd} ({grd_counts.iloc[0]:,} egresos)")
print(f"Top 5 GRDs:")
print(grd_counts.head(5).to_string())
print()

df_grd = df_clean[df_clean["IR_29301_COD_GRD"] == target_grd].copy()

# Hospitales con al menos 50 casos de este GRD
hosp_counts = df_grd["COD_HOSPITAL"].value_counts()
valid_hosps = hosp_counts[hosp_counts >= 50].index
df_grd_valid = df_grd[df_grd["COD_HOSPITAL"].isin(valid_hosps)]

print(f"Hospitales con ≥50 casos de este GRD: {len(valid_hosps)}")
print(f"Registros para análisis: {len(df_grd_valid):,}")


In [ ]:
# Estadística descriptiva por hospital para el GRD focal
summary_hosp = (
    df_grd_valid.groupby("COD_HOSPITAL")
    .agg(
        n=("dias_estada", "size"),
        media=("dias_estada", "mean"),
        mediana=("dias_estada", "median"),
        desv_std=("dias_estada", "std"),
        p25=("dias_estada", lambda x: x.quantile(0.25)),
        p75=("dias_estada", lambda x: x.quantile(0.75)),
    )
    .reset_index()
    .sort_values("media", ascending=False)
)

# Coeficiente de variación entre hospitales
cv_between = summary_hosp["media"].std() / summary_hosp["media"].mean() * 100
print(f"Coeficiente de variación ENTRE hospitales (media de días): {cv_between:.1f}%")
print(f"Rango de medias: {summary_hosp['media'].min():.1f} – {summary_hosp['media'].max():.1f} días")
print()
display(summary_hosp.head(15).round(2))


In [ ]:
# Boxplot de días de estada por hospital (top 20 por volumen)
top20_hosp = hosp_counts[hosp_counts >= 50].head(20).index
df_plot_hosp = df_grd_valid[df_grd_valid["COD_HOSPITAL"].isin(top20_hosp)].copy()

# Ordenar por mediana
order = (df_plot_hosp.groupby("COD_HOSPITAL")["dias_estada"]
         .median().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(16, 8))
sns.boxplot(data=df_plot_hosp, x="COD_HOSPITAL", y="dias_estada",
            order=order, showfliers=False, palette="viridis", ax=ax, width=0.7)
ax.axhline(df_grd_valid["dias_estada"].median(), color="red", linestyle="--",
           linewidth=2, alpha=0.7, label=f"Mediana global: {df_grd_valid['dias_estada'].median():.0f} días")
ax.set_xlabel("Código de hospital")
ax.set_ylabel("Días de estada")
ax.set_title(f"Variabilidad en días de estada por hospital — GRD {target_grd} (mismo diagnóstico)")
ax.legend(fontsize=10)
plt.xticks(rotation=45, ha="right", fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 8.1 — Evidencia de variabilidad hospitalaria:**  
Para el mismo código GRD (es decir, pacientes con diagnóstico y complejidad comparables según la clasificación IR-GRD), se observan diferencias notables entre hospitales tanto en la mediana como en la dispersión de los días de estada. Algunos hospitales presentan estancias sistemáticamente más largas que otros, lo que sugiere que factores organizacionales, de capacidad o de práctica clínica local influyen en el resultado, más allá de la condición del paciente. El coeficiente de variación entre hospitales cuantifica esta heterogeneidad.


### 8.2 Variabilidad por Servicio de Salud

In [ ]:
# Analizar días de estada promedio por servicio de salud
ss_stats = (
    df_clean.groupby("SERVICIO_SALUD")
    .agg(
        n=("dias_estada", "size"),
        media=("dias_estada", "mean"),
        mediana=("dias_estada", "median"),
    )
    .reset_index()
    .sort_values("media", ascending=False)
)
ss_stats = ss_stats[ss_stats["n"] >= 1000]  # Solo servicios con volumen relevante

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(ss_stats["SERVICIO_SALUD"][::-1], ss_stats["media"][::-1],
        color=COLORES[0], edgecolor="white", alpha=0.85)
ax.axvline(df_clean["dias_estada"].mean(), color="red", linestyle="--",
           linewidth=2, label=f"Media nacional: {df_clean['dias_estada'].mean():.2f}")
for i, (_, row) in enumerate(ss_stats[::-1].iterrows()):
    ax.text(row["media"] + 0.1, i, f"{row['media']:.1f}", va="center", fontsize=8)

ax.set_xlabel("Media de días de estada")
ax.set_title("Estancia media por Servicio de Salud")
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()


**Conclusión 8.2 — Variabilidad geográfica:**  
La estancia media varía significativamente entre Servicios de Salud del país. Estas diferencias pueden reflejar distintos perfiles epidemiológicos regionales, diferencias en la disponibilidad de camas y recursos, o variaciones en las prácticas clínicas locales. Los servicios de salud con estancias consistentemente por encima de la media nacional podrían beneficiarse de intervenciones de gestión clínica (protocolos de alta temprana, cirugía ambulatoria, gestión de camas).


### 8.3 Prueba estadística: Kruskal-Wallis

In [ ]:
# Test de normalidad (Shapiro-Wilk en submuestra)
alpha = 0.05
x_sample = df_grd_valid["dias_estada"].dropna().values
rng = np.random.default_rng(42)
x_norm = rng.choice(x_sample, size=min(5000, len(x_sample)), replace=False)
w_stat, p_norm = stats.shapiro(x_norm)

print("=" * 60)
print("TEST DE NORMALIDAD (Shapiro-Wilk, submuestra n=5000)")
print("=" * 60)
print(f"Estadístico W: {w_stat:.6f}")
print(f"P-value:       {p_norm:.2e}")
print(f"Resultado:     {'Se RECHAZA H0 de normalidad' if p_norm < alpha else 'No se rechaza H0'}")
print(f"               → Se justifica usar prueba no paramétrica (Kruskal-Wallis)")
print()

# Kruskal-Wallis
groups = [g["dias_estada"].dropna().values for _, g in df_grd_valid.groupby("COD_HOSPITAL")]
h_stat, p_kw = stats.kruskal(*groups)

print("=" * 60)
print("TEST DE KRUSKAL-WALLIS (comparación entre hospitales)")
print("=" * 60)
print(f"GRD analizado:          {target_grd}")
print(f"Hospitales comparados:  {len(groups)}")
print(f"Estadístico H:          {h_stat:.4f}")
print(f"P-value:                {p_kw:.2e}")
print(f"Alpha:                  {alpha}")
print()
if p_kw < alpha:
    print("CONCLUSIÓN: Se RECHAZA H0.")
    print("Existen diferencias ESTADÍSTICAMENTE SIGNIFICATIVAS en los días de")
    print("estada entre hospitales, para pacientes con el mismo GRD.")
else:
    print("CONCLUSIÓN: No se rechaza H0.")
    print("No hay evidencia suficiente de diferencias entre hospitales.")

# Effect size (Epsilon-squared)
N = sum(len(g) for g in groups)
k = len(groups)
eps_sq = (h_stat - k + 1) / (N - k)
print(f"\nTamaño del efecto (ε²): {eps_sq:.4f}")
if eps_sq < 0.01:
    print("Interpretación: Efecto pequeño")
elif eps_sq < 0.06:
    print("Interpretación: Efecto moderado")
else:
    print("Interpretación: Efecto grande")


**Conclusión 8.3 — Significancia estadística de la variabilidad:**  
El test de Shapiro-Wilk confirma que la distribución de días de estada no es normal, lo que justifica el uso de la prueba no paramétrica de Kruskal-Wallis. Esta prueba revela que las diferencias entre hospitales son **estadísticamente significativas** (p < 0.05), lo que respalda la hipótesis de que el hospital donde se atiende un paciente influye en su estancia, incluso cuando se controla por diagnóstico mediante el código GRD. El tamaño del efecto (epsilon-cuadrado) complementa la significancia estadística con una medida de la magnitud práctica de esta diferencia.


### 8.4 Intervalos de Confianza al 95% por Hospital

Para cada hospital con al menos 30 casos del GRD focal se construye un **intervalo de confianza al 95%** para la media de días de estada usando la distribución t de Student. Esta visualización complementa el boxplot (sección 8.1): en lugar de mostrar la distribución completa, enfoca la atención en la *estimación de la media* y su precisión.

In [ ]:
from scipy.stats import t as t_dist

# Intervalos de confianza al 95% para la media de días de estada por hospital
alpha_ic = 0.05
MIN_CASOS_IC = 30
ic_list = []

for hosp, grupo in df_grd_valid.groupby("COD_HOSPITAL"):
    x = grupo["dias_estada"].dropna()
    n = len(x)
    if n < MIN_CASOS_IC:
        continue
    media = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)
    t_crit = t_dist.ppf(1 - alpha_ic / 2, df=n - 1)
    ic_list.append({
        "Hospital": str(hosp),
        "n_casos": n,
        "Media": round(media, 2),
        "IC_inferior": round(media - t_crit * se, 2),
        "IC_superior": round(media + t_crit * se, 2),
        "Amplitud_IC": round(2 * t_crit * se, 2),
    })

df_ic = pd.DataFrame(ic_list).sort_values("Media", ascending=False).reset_index(drop=True)
media_global = df_grd_valid["dias_estada"].mean()

print(f"Intervalos de Confianza al {(1 - alpha_ic)*100:.0f}% — Días de Estada (GRD = {target_grd})")
print(f"Hospitales con >= {MIN_CASOS_IC} casos: {len(df_ic)}")
print(f"Media global: {media_global:.2f} días\n")
display(df_ic.head(20))

# Gráfico: top 25 hospitales por volumen, ordenados por media
df_ic_plot = df_ic.nlargest(25, "n_casos").sort_values("Media")

fig, ax = plt.subplots(figsize=(14, 10))
y_pos = range(len(df_ic_plot))

ax.barh(
    y_pos, df_ic_plot["Media"],
    xerr=[
        df_ic_plot["Media"] - df_ic_plot["IC_inferior"],
        df_ic_plot["IC_superior"] - df_ic_plot["Media"]
    ],
    color=COLORES[0], alpha=0.65, edgecolor="white",
    error_kw={"capsize": 5, "elinewidth": 2, "capthick": 2, "ecolor": "gray"}
)
ax.axvline(media_global, color="red", linestyle="--", linewidth=2,
           label=f"Media global: {media_global:.2f} días")
ax.set_yticks(y_pos)
ax.set_yticklabels(df_ic_plot["Hospital"], fontsize=8)
ax.set_xlabel("Días de Estada (media ± IC 95%)")
ax.set_title(
    f"Intervalos de Confianza al 95% por Hospital — GRD: {target_grd}\n"
    f"(Top 25 hospitales por volumen, con ≥{MIN_CASOS_IC} casos)"
)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

**Conclusión 8.4 — Intervalos de Confianza por hospital:**  
Los intervalos de confianza al 95% para la media de días de estada por hospital ofrecen una representación geométrica de la variabilidad. Hospitales cuyos intervalos **no se solapan** presentan diferencias estadísticamente significativas en su estancia media. La línea de media global es la referencia: establecimientos consistentemente por encima de ella tienen estancias más largas de lo esperado para el mismo diagnóstico. La amplitud del intervalo refleja también la incertidumbre: hospitales con pocos casos tienen intervalos más amplios, mientras que los de mayor volumen permiten estimaciones más precisas.

### 8.5 Análisis Post-hoc: Test de Dunn

El test de Kruskal-Wallis (sección 8.3) solo indica si **existe al menos un par** de hospitales con diferencias significativas. El test de Dunn (con corrección de Bonferroni para comparaciones múltiples) permite identificar **qué pares específicos** de hospitales difieren, y en qué dirección. El mapa de calor resultante muestra la topología de la red: hospitales similares entre sí versus establecimientos con prácticas claramente distintas.

In [ ]:
from itertools import combinations
import matplotlib.colors as mcolors

# ── Test de Dunn post-hoc (Kruskal-Wallis) con corrección de Bonferroni ──
MIN_CASES_DUNN = 50

hosp_dunn_counts = df_grd_valid["COD_HOSPITAL"].value_counts()
hosp_dunn = hosp_dunn_counts[hosp_dunn_counts >= MIN_CASES_DUNN].index.tolist()
df_dunn_data = df_grd_valid[df_grd_valid["COD_HOSPITAL"].isin(hosp_dunn)].copy()

groups_dunn = {
    str(hosp): grupo["dias_estada"].dropna().values
    for hosp, grupo in df_dunn_data.groupby("COD_HOSPITAL")
}


def dunn_bonferroni(groups_dict):
    """Test de Dunn con corrección de Bonferroni para comparaciones múltiples."""
    all_data = np.concatenate(list(groups_dict.values()))
    ranks = stats.rankdata(all_data)
    N = len(all_data)

    sizes, rank_sums = {}, {}
    start = 0
    for k, v in groups_dict.items():
        end = start + len(v)
        sizes[k] = len(v)
        rank_sums[k] = ranks[start:end].sum()
        start = end

    pairs = list(combinations(groups_dict.keys(), 2))
    n_comp = len(pairs)
    rows = []

    for g1, g2 in pairs:
        n1, n2 = sizes[g1], sizes[g2]
        mr1 = rank_sums[g1] / n1
        mr2 = rank_sums[g2] / n2
        se = np.sqrt((N * (N + 1) / 12) * (1 / n1 + 1 / n2))
        z = abs(mr1 - mr2) / se
        p_raw = 2 * (1 - stats.norm.cdf(z))
        p_adj = min(p_raw * n_comp, 1.0)
        rows.append({"Hospital_A": g1, "Hospital_B": g2,
                     "Z": round(z, 4), "p_valor": round(p_raw, 6),
                     "p_ajustado_Bonferroni": round(p_adj, 6)})

    return pd.DataFrame(rows).sort_values("p_ajustado_Bonferroni")


alpha = 0.05
df_dunn_res = dunn_bonferroni(groups_dunn)
sig_pairs = df_dunn_res[df_dunn_res["p_ajustado_Bonferroni"] < alpha]

print(f"Test de Dunn post-hoc — GRD: {target_grd}")
print(f"Hospitales comparados: {len(hosp_dunn)}")
print(f"Pares totales: {len(df_dunn_res)}")
print(f"Pares con diferencia significativa (p_Bonf < {alpha}): {len(sig_pairs)}")
print(f"\nTop 20 pares más significativos:")
display(df_dunn_res.head(20))

# Heatmap de significancia (top 15 hospitales por volumen)
top15_dunn = [str(h) for h in hosp_dunn[:15]]
sig_matrix = pd.DataFrame(np.ones((15, 15)), index=top15_dunn, columns=top15_dunn)

for _, row in df_dunn_res.iterrows():
    if row["Hospital_A"] in top15_dunn and row["Hospital_B"] in top15_dunn:
        sig_matrix.loc[row["Hospital_A"], row["Hospital_B"]] = row["p_ajustado_Bonferroni"]
        sig_matrix.loc[row["Hospital_B"], row["Hospital_A"]] = row["p_ajustado_Bonferroni"]

sig_binary = (sig_matrix < alpha).astype(float)
np.fill_diagonal(sig_binary.values, np.nan)

fig, ax = plt.subplots(figsize=(12, 10))
cmap_sig = mcolors.ListedColormap(["#d4edda", "#f8d7da"])
sns.heatmap(sig_binary, annot=False, cmap=cmap_sig, ax=ax,
            linewidths=0.5,
            cbar_kws={"label": "1 = diferencia significativa (p_Bonf < 0.05)"})
ax.set_title(
    f"Pares de hospitales con diferencia significativa — Dunn (Bonferroni)\n"
    f"GRD: {target_grd} | Verde = similar, Rojo = diferente"
)
ax.set_xlabel("Hospital")
ax.set_ylabel("Hospital")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

**Conclusión 8.5 — Análisis post-hoc Dunn:**  
El test de Dunn con corrección de Bonferroni identifica qué **pares específicos** de hospitales presentan diferencias estadísticamente significativas en días de estada. A diferencia de Kruskal-Wallis (que solo indica si *algún* par difiere), Dunn permite priorizar establecimientos para intervenciones de mejora. El mapa de calor de significancia visualiza la topología de las diferencias: clusters de hospitales similares vs. establecimientos sistemáticamente distintos al resto. Este análisis es el paso esencial entre detectar variabilidad (sección 8.3) y actuar sobre ella.

### 8.6 Consistencia de la Variabilidad en Múltiples GRDs

En lugar de analizar solo el GRD más frecuente, esta sección extiende el análisis a los **5 GRDs con mayor volumen** de casos. Para cada hospital se calcula el *ratio* entre su media de días de estada y la media nacional del GRD. Un ratio > 1 indica que ese hospital tiene estancias más largas que el promedio; un ratio < 1 indica mayor eficiencia. La consistencia de estos ratios a través de múltiples GRDs es evidencia de patrones institucionales.

In [ ]:
# Variabilidad hospitalaria consistente a través de los 5 GRDs más frecuentes
top5_grds = df_clean["IR_29301_COD_GRD"].value_counts().head(5).index.tolist()

results_multi = []
for grd in top5_grds:
    df_g = df_clean[df_clean["IR_29301_COD_GRD"] == grd].copy()
    media_grd = df_g["dias_estada"].mean()
    if media_grd == 0:
        continue

    hosp_stats = (
        df_g.groupby("COD_HOSPITAL")
        .agg(n=("dias_estada", "size"), media=("dias_estada", "mean"))
        .reset_index()
    )
    hosp_stats = hosp_stats[hosp_stats["n"] >= 30]
    hosp_stats["ratio"] = hosp_stats["media"] / media_grd
    hosp_stats["GRD"] = str(grd)
    results_multi.append(hosp_stats)

df_multi = pd.concat(results_multi, ignore_index=True)

# Hospitales presentes en al menos 4 de los 5 GRDs
hosp_coverage = df_multi.groupby("COD_HOSPITAL")["GRD"].nunique()
hosp_consistent = hosp_coverage[hosp_coverage >= 4].index

df_pivot = df_multi[df_multi["COD_HOSPITAL"].isin(hosp_consistent)].pivot_table(
    index="COD_HOSPITAL", columns="GRD", values="ratio"
)
df_pivot["media_ratio"] = df_pivot.mean(axis=1)
df_pivot = df_pivot.sort_values("media_ratio", ascending=False)

# Plot: top 20 hospitales con mayor variación
top20_multi = df_pivot.head(20).drop(columns="media_ratio")

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    top20_multi,
    annot=True, fmt=".2f",
    cmap="RdYlGn_r",
    center=1.0, vmin=0.5, vmax=2.0,
    linewidths=0.4,
    cbar_kws={"label": "Ratio días estada / media nacional GRD"},
    ax=ax
)
ax.set_title(
    "Ratio de días de estada por hospital y GRD (Top 20 hospitales con estancias más largas)\n"
    "Ratio > 1 = estada mayor que la media nacional del GRD"
)
ax.set_xlabel("GRD")
ax.set_ylabel("Hospital")
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

print(f"Hospitales con datos en >= 4 de los 5 GRDs: {len(hosp_consistent)}")
print(f"\nTop 5 hospitales con mayor ratio promedio (estancias más largas):")
display(df_pivot.head(5).round(2))
print(f"\nTop 5 hospitales con menor ratio promedio (estancias más eficientes):")
display(df_pivot.tail(5).round(2))

**Conclusión 8.6 — Consistencia multi-GRD de la variabilidad:**  
Extender el análisis a los 5 GRDs más frecuentes permite evaluar si la variabilidad hospitalaria es un fenómeno robusto o específico de un diagnóstico. Un hospital con ratios sistemáticamente superiores a 1 en múltiples GRDs evidencia un patrón organizacional de estancias prolongadas, no explicable por el perfil clínico de un diagnóstico en particular. Por el contrario, hospitales con ratios cercanos a 1 de forma consistente sugieren una gestión más eficiente y estandarizada. Esta consistencia multi-GRD refuerza la hipótesis de que las diferencias observadas responden a factores institucionales, culturales o de recursos propios de cada establecimiento.

## 9. Conclusiones Generales

### Hallazgos principales

1. **Impacto pandémico y recuperación:** La pandemia de COVID-19 causó una reducción drástica en los egresos hospitalarios durante 2020. La recuperación fue progresiva y para 2023-2024 el sistema ya superó los niveles pre-pandemia, sugiriendo tanto resolución de demanda acumulada como crecimiento orgánico.

2. **Perfil demográfico:** El sistema público hospitalario chileno atiende predominantemente a mujeres en edad reproductiva (carga obstétrica) y adultos mayores de ambos sexos (carga crónica). La población beneficiaria es mayoritariamente FONASA A y B (tramos de mayor vulnerabilidad socioeconómica). La presencia de pueblos originarios, aunque minoritaria, es relevante para políticas de salud intercultural.

3. **Perfil clínico:** Las causas de hospitalización más frecuentes son obstétricas, quirúrgicas abdominales (colecistitis), oftalmológicas (cataratas) y oncológicas. La coexistencia de patología aguda y crónica refleja la transición epidemiológica del país.

4. **Estancia hospitalaria:** La distribución de días de estada es altamente asimétrica, con una mediana corta (~2 días) pero una cola larga de estancias prolongadas. La severidad del episodio es el principal determinante clínico de la duración de la estancia.

5. **Variabilidad hospitalaria comprobada (GRD focal):** Se demostró estadísticamente (Kruskal-Wallis, p < 0.05) que existen diferencias significativas en los días de estada entre hospitales para pacientes con el mismo GRD. Los intervalos de confianza al 95% visualizan esta heterogeneidad: varios hospitales tienen medias que no se solapan con la media global.

6. **Pares de hospitales significativamente distintos (Dunn):** El análisis post-hoc de Dunn con corrección de Bonferroni identifica los pares específicos de hospitales con diferencias estadísticamente significativas, revelando la topología de la variabilidad: no todos los hospitales difieren entre sí, pero existen establecimientos con prácticas sistemáticamente distintas al resto de la red.

7. **Variabilidad consistente en múltiples GRDs:** Los ratios de estada por hospital se mantienen relativamente estables a través de los 5 GRDs más frecuentes, lo que indica que la variabilidad es un fenómeno institucional, no diagnóstico-específico. Hospitales con ratios consistentemente altos o bajos exhiben patrones organizacionales y de práctica clínica propios.

8. **Estacionalidad predecible:** Los egresos siguen un patrón estacional consistente (pico invernal, valle estival), lo que permite una planificación anticipada de recursos.

---

### Limitaciones

- Los datos GRD clasifican por diagnóstico y severidad, pero no capturan todas las variables relevantes (comorbilidades detalladas, estado funcional, preferencias del paciente).
- La calidad del registro varía entre establecimientos, lo que puede introducir sesgos de clasificación.
- El análisis es descriptivo e inferencial básico; no establece causalidad entre hospital y resultado.
- No se dispone de variables de resultado clínico final como reingresos, complicaciones postoperatorias o satisfacción del paciente.
- Los GRDs con pocos casos por hospital producen intervalos de confianza amplios, limitando la precisión de las comparaciones.

---

### Líneas futuras de investigación

- **Modelos multinivel (mixed-effects):** Aislar el efecto hospital ajustando simultáneamente por severidad, edad, sexo y peso GRD. El coeficiente del nivel hospital cuantifica la variabilidad "inexplicada" por el paciente.
- **Análisis de eficiencia:** Comparar el peso GRD producido versus los recursos consumidos (camas-día) para identificar establecimientos con alta o baja eficiencia técnica.
- **Estudio longitudinal pandémico:** Evaluar el impacto del COVID-19 en las listas de espera quirúrgicas y la composición del case-mix por año.
- **Análisis por especialidad:** Identificar áreas clínicas con mayor variabilidad injustificada (ej.: cirugía, traumatología, medicina interna).
- **Incorporación de datos de resultados:** Reingresos a 30 días, mortalidad ajustada por riesgo, complicaciones para una evaluación más completa de la calidad asistencial.
- **Pertinencia cultural:** Analizar diferencias en estancia y procedimientos según etnia del paciente, controlando por severidad y diagnóstico.